# The Stroke Visualiser

Data is abstract until you can *see* it.

This notebook builds a visualiser that takes a stroke-3 array and animates it stroke by stroke.
This is **Checkpoint 1: The Stroke Engine**.

## 0. Setup

This cell redefines the helpers from the data notebook so this notebook is fully self-contained.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def drawing_to_stroke3(drawing):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

def stroke3_to_absolute(stroke3):
    abs_coords = np.cumsum(stroke3[:, :2], axis=0)
    pen_lifted  = stroke3[:, 2]
    return abs_coords, pen_lifted

drawings = []
with open('data/cat.ndjson') as f:
    for line in f:
        drawings.append(json.loads(line))
        if len(drawings) >= 20:
            break

print(f'Loaded {len(drawings)} drawings.')


## 1. Converting deltas back to absolute coordinates

To plot, we need absolute `(x, y)` positions. We get them by cumulatively summing the deltas.

In [ ]:
s3 = drawing_to_stroke3(drawings[0]['drawing'])
coords, pen_lifted = stroke3_to_absolute(s3)

print('stroke-3 shape   :', s3.shape)
print('abs coords shape :', coords.shape)
print()
print('stroke-3 first 5 rows (dx, dy, pen):')
print(s3[:5])
print()
print('absolute coords first 5 rows (x, y):')
print(coords[:5].round(1))

# TODO: What is np.cumsum doing exactly?
#       Manually compute coords[3] from s3[:4, :2] and verify it matches.
# YOUR CODE HERE

# np.cumsum computes the cumulative sum of the elements along a given axis.
# By summing the dx and dy values at each step, we reconstruct the absolute (x, y) path of the pen.

val = np.sum(s3[:4, :2], axis=0)
print(val)
print(coords[3])
# They match.

## 2. Static plot

We split the array into individual strokes wherever `pen_lifted == 1`, then plot each segment.

In [ ]:
def plot_sketch(stroke3, title='Sketch', color='black', ax=None):
    coords, pen_lifted = stroke3_to_absolute(stroke3)

    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))

    ax.set_aspect('equal')
    ax.invert_yaxis()   # y increases downward in screen coordinates
    ax.axis('off')
    ax.set_title(title)

    start = 0
    for i in range(len(pen_lifted)):
        if pen_lifted[i] == 1:
            segment = coords[start : i + 1]
            ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=2)
            start = i + 1

s3 = drawing_to_stroke3(drawings[0]['drawing'])
plot_sketch(s3, title='Cat drawing 0')
plt.show()


In [ ]:
# TODO: Plot drawings[0] through drawings[7] in a 2x4 grid.
#       Use fig, axes = plt.subplots(2, 4, figsize=(14, 8))
#       Give each subplot the title 'Cat N' where N is the drawing index.
# YOUR CODE HERE

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
axes = axes.flatten()

for i in range(8):
    s3_n = drawing_to_stroke3(drawings[i]['drawing'])
    plot_sketch(s3_n, title=f'Cat {i}', ax=axes[i])

plt.tight_layout()
plt.show()


## 3. Animated visualiser

We draw one point at a time using `FuncAnimation`, pausing briefly between frames.

In [ ]:
def animate_sketch(stroke3, interval=25):
    '''
    Animate a stroke-3 sketch, drawing one point at a time.

    stroke3  : numpy array shape (N, 3)
    interval : milliseconds between frames (lower = faster)
    '''
    coords, pen_lifted = stroke3_to_absolute(stroke3)
    n = len(coords)

    x_min, x_max = coords[:, 0].min() - 5, coords[:, 0].max() + 5
    y_min, y_max = coords[:, 1].min() - 5, coords[:, 1].max() + 5

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_aspect('equal')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_max, y_min)   # inverted
    ax.axis('off')

    cur_x, cur_y = [], []

    def update(frame):
        cur_x.append(coords[frame, 0])
        cur_y.append(coords[frame, 1])

        if len(cur_x) > 1:
            ax.plot(cur_x[-2:], cur_y[-2:], 'k-', linewidth=2)

        if pen_lifted[frame] == 1:
            cur_x.clear()
            cur_y.clear()

    anim = FuncAnimation(fig, update, frames=n, interval=interval, repeat=False)
    plt.close(fig)
    return anim

s3   = drawing_to_stroke3(drawings[0]['drawing'])
anim = animate_sketch(s3, interval=20)
HTML(anim.to_jshtml())


In [ ]:
# TODO: Animate drawings[3]. Does it look like a cat?
#       Try interval=5 and interval=100. What changes and why?
# YOUR CODE HERE

s3_3 = drawing_to_stroke3(drawings[3]['drawing'])

anim_5 = animate_sketch(s3_3, interval=5)
anim_100 = animate_sketch(s3_3, interval=100)

display(HTML(anim_5.to_jshtml()))
display(HTML(anim_100.to_jshtml()))

# It looks like a simple line drawing of a cat.
# The interval parameter changes the delay in milliseconds between each frame of the animation.
# interval=5 makes the animation draw very quickly, while interval=100 makes it draw very slowly point by point.


## 4. Visualising model output (Week 3 preview)

In Week 3 the decoder will output stroke-3 tensors. We will call `plot_tensor_sketch` to check whether the reconstructions look right.

In [ ]:
import torch

def plot_tensor_sketch(stroke3_tensor, title='Model output'):
    '''
    Same as plot_sketch but accepts a PyTorch tensor.
    Rounds the pen_lifted column to 0 or 1 since model outputs are continuous.
    '''
    s3 = stroke3_tensor.detach().cpu().numpy()
    s3[:, 2] = (s3[:, 2] > 0.5).astype(np.float32)
    plot_sketch(s3, title=title)
    plt.show()

# Using real data as a stand-in for model output
fake_output = torch.tensor(drawing_to_stroke3(drawings[1]['drawing']))
plot_tensor_sketch(fake_output, title='Simulated model output')
print('In Week 3, we replace this with actual decoder outputs.')


## Checkpoint 1 complete!

You can now:
- Load and parse Quick Draw `.ndjson` files
- Convert to stroke-3 delta format
- Plot static sketches from delta arrays
- Animate sketches stroke by stroke
- Visualise PyTorch tensor outputs

Next: `assignment/assignment.ipynb`